# BM25 Retrieval Engine

***Step 2 — Install Dependencies***

In [17]:
pip install rank-bm25 nltk

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


***Step 3 — Load Cleaned Dataset***

In [ ]:
import pandas as pd

print("Loading cleaned dataset...\n")

df = pd.read_csv(
    "data/processed/cleaned_products1.csv"
)

print(df.head())

print("\nDataset Shape:")
print(df.shape)

Loading cleaned dataset...

         asin                                              title  rating  \
0  B08VJFZQ9S  प्लेन कैज़ुअल वियर बेसबॉल कैप पुरुषों और महिला...     0.0   
1  B08VJFYW5Q  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, फ़्...     0.0   
2  B08VJFYVX9  प्लेन कैज़ुअल वियर बेसबॉल कैप पुरुषों और महिला...     0.0   
3  B08VJFXM7F  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (सफ़ेद, फ़...     0.0   
4  B08VJFXFTJ  यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (लाल और का...     0.0   

   review_count  price  listPrice                   category  isBestSeller  \
0             0  299.0      499.0  पुरुषों के हैट्स और कैप्स         False   
1             0  299.0      499.0  पुरुषों के हैट्स और कैप्स         False   
2             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   
3             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   
4             0  275.0      300.0  पुरुषों के हैट्स और कैप्स         False   

   boughtInLastMonth                          

In [19]:
from rank_bm25 import BM25Okapi

from nltk.tokenize import word_tokenize

import nltk

In [20]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

***Step 5 — Download Tokenizer***

In [21]:
import nltk

nltk.download('punkt')

nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\gagan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

***Step 6 — Tokenize Search Corpus***

In [22]:
print("Tokenizing search corpus...\n")

tokenized_corpus = [
    word_tokenize(text.lower())
    for text in df["search_text"]
]

print("Tokenization completed!")

Tokenizing search corpus...

Tokenization completed!


***Step 7 — Build BM25 Index***

In [23]:
from rank_bm25 import BM25Okapi

print("Building BM25 index...\n")

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created successfully!")

Building BM25 index...

BM25 index created successfully!


***Step 8 — Test Search Queries***

In [24]:
query = "कैप"

tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)

print(scores[:5])

[2.89056933 2.41801554 3.01460489 2.41801554 2.31191861]


***Step 9 — Retrieve Top Ranked Products***

In [25]:
import numpy as np

# Get indices of top 5 products
top_n = np.argsort(scores)[::-1][:5]

# Retrieve products
results = df.iloc[top_n]

# Display results
print(
    results[
        [
            "title",
            "price",
            "rating"
        ]
    ]
)

                                                  title  price  rating
2084  Bhagwati Store कैप कैप कैप पुरुषों के लिए महिल...    0.0     0.0
2794  लड़कियों के लिए पीला और काला प्रिंटेड एंगोरा क...  475.0     0.0
2125  Bhagwati Store कैप मिक्स मल्टीकलर कैप गोल्फ कै...    0.0     0.0
2118  Bhagwati Store कैप मिक्स मल्टीकलर कैप गोल्फ कै...    0.0     0.0
2577  यूनीसेक्स विंटर कैप पुरुषों और विंटर कैप महिला...  279.0     3.4


In [26]:
df = df[
    (df["price"] > 0) &
    (df["rating"] >= 0)
]

***Step 10 — Create Reusable Search Function***

In [27]:
import numpy as np

def search_products(query, top_k=5):

    # Tokenize query
    tokenized_query = query.lower().split()

    # Get BM25 scores
    scores = bm25.get_scores(tokenized_query)

    # Get top product indices
    top_n = np.argsort(scores)[::-1][:top_k]

    # Retrieve products
    results = df.iloc[top_n]

    return results[
        [
            "title",
            "price",
            "rating"
        ]
    ]

***Testing***

In [28]:
search_products("कैप")

,title,price,rating
2114,पुरुषों के लिए अनोखा डिज़ाइन स्पोर्ट्स स्नैपबै...,499.0,0.0
2836,पुरुषों के लिए ट्रेडिशनल कॉटन सिंगल धोती लंज ब...,374.0,3.9
2167,"हैंडलूम यूनीसेक्स कुल्लू पट्टी (पहरी कैप, हीमा...",390.0,0.0
2160,पुरुषों के लिए यूनीसेक्स कॉटन स्नैपबैक बेसबॉल ...,399.0,3.5
2619,विंटर यूनीसेक्स ऊनी बीनी कैप -1 (फ़्री साइज़),499.0,4.0


In [29]:
search_products("जूते")

,title,price,rating
16,यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला और स...,299.0,0.0
17,"यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, लाल...",399.0,0.0
18,यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (लाल और सफ...,375.0,0.0
19,Unisex Cotton Adjustable Baseball Cap (Red & W...,299.0,0.0
20,"यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, लाल...",399.0,0.0


In [30]:
search_products("घड़ी")

,title,price,rating
16,यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला और स...,299.0,0.0
17,"यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, लाल...",399.0,0.0
18,यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (लाल और सफ...,375.0,0.0
19,Unisex Cotton Adjustable Baseball Cap (Red & W...,299.0,0.0
20,"यूनीसेक्स कॉटन एडजस्टेबल बेसबॉल कैप (काला, लाल...",399.0,0.0


In [31]:
search_products("बैग")

,title,price,rating
5676,St.Botanica Pro Keratin और Argan तेल चिकना थेर...,251.0,3.9
17949,"JUZZII फेशियल स्पा हेडबैंड, एडजस्टेबल इलास्टिक...",249.0,4.1
6605,Vebix Professional अनुकूलित हेयर फॉल कंट्रोल र...,679.0,0.0
8983,Abrish 5 क्लिप्स आधारित 24 इंच वेवी/कर्ली सिंथ...,249.0,5.0
7302,"Tikitoro Kids Nourishing Hair Oil, 100% Vegan ...",428.0,3.5


***Step 12 — Filter Low Quality Products***

In [32]:
df = df[
    (df["price"] > 0)
]

print(df.shape)

(19853, 10)


In [33]:
df = df[
    (df["price"] > 0) &
    (df["review_count"] >= 0)
]

print(df.shape)

(19853, 10)


In [35]:
# Rebuilding Tokenized Corpus
from nltk.tokenize import word_tokenize

tokenized_corpus = [
    word_tokenize(text.lower())
    for text in df["search_text"]
]

In [36]:
# Rebuilding BM25 Index
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index rebuilt successfully!")

BM25 index rebuilt successfully!


***Test Queries Again***

In [37]:
search_products("कैप")

,title,price,rating
2794,लड़कियों के लिए पीला और काला प्रिंटेड एंगोरा क...,475.0,0.0
2577,यूनीसेक्स विंटर कैप पुरुषों और विंटर कैप महिला...,279.0,3.4
2424,लड़कों के लिए डिज़ाइनर बीनी कैप | Angoora कैप ...,345.0,0.0
2430,पुरुषों के लिए डिज़ाइनर बीनी कैप | Angoora कैप...,345.0,0.0
2158,Kids's I Love Panda सफ़ेद और काला रंग कैप हाफ ...,299.0,0.0


In [38]:
search_products("जूते")

,title,price,rating
0,प्लेन कैज़ुअल वियर बेसबॉल कैप पुरुषों और महिला...,299.0,0.0
19999,INHEAVEN 3.5 इंच हेयर क्लॉ 3 PC क्लिप लार्ज नो...,189.0,2.8
19998,Petal Fresh शुद्ध एंटी-फ्रिज़ लैवेंडर शैम्पू |...,684.0,3.4
19997,"Schwarzkopf टैफ्ट अल्ट्रा हेयर वैक्स, गीले और ...",350.0,0.0
19996,ऑनलाइन गुणवत्ता स्टोर ड्राई शैंपू और कंडीशनर.,55.0,3.7


In [39]:
search_products("बैग")

,title,price,rating
5627,36 स्ट्रिप्स का कोई शाइन टेप CC कंटूर बैग नहीं,415.0,3.7
17823,Brustro कलाकार मिश्रित बाल ब्रश पु बैग में 15 ...,1392.0,4.4
6551,"Walker Tape अल्ट्रा होल्ड मिनी टेप स्ट्रिप्स, ...",790.0,3.7
8918,Sevich All Hair Building फाइबर रीफिल बैग (काला...,699.0,4.1
7244,Sevich All Hair Building फाइबर रीफिल बैग (काला...,624.0,3.7
